In [ ]:
import torch
torch.__version__

In [ ]:
import math
import torch.nn as nn

In [ ]:
batch_size = 4
seq_len = 100
embed_dim = 512

num_heads = 8
head_dim = embed_dim // num_heads #64

X = torch.randn(batch_size, seq_len, embed_dim) #input

# weight matrices to get Qs, Ks, Vs (batched)
Wq_batched = torch.randn(num_heads, embed_dim, head_dim)
Wk_batched = torch.randn(num_heads, embed_dim, head_dim)
Wv_batched = torch.randn(num_heads, embed_dim, head_dim)

# Q = X @ Wq (batched and boradcasted)
Q_batched = torch.matmul(X.unsqueeze(1), Wq_batched.unsqueeze(0))
K_batched = torch.matmul(X.unsqueeze(1), Wk_batched.unsqueeze(0))
V_batched = torch.matmul(X.unsqueeze(1), Wv_batched.unsqueeze(0))
# shape = [batch_size, num_heads, seq_len, head_dim]


# Attention
# Q \bt (num_seq x head_dim)
# z = softmax(Q *K.t) V / sqrt(head_dim)
# batched matmul
S = torch.matmul(Q_batched, torch.transpose(K_batched,2,3))
S = S / math.sqrt(head_dim) #[batch_size, num_heads, seq_len, seq_len]

#mask
mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
S = S.masked_fill(~mask, float('-inf'))

P = torch.softmax(S, dim=-1) #[batch_size, num_heads, seq_len, seq_len]
O = torch.matmul(P, V_batched) #[batch_size, num_heads, seq_len, head_dim]
# reshape/permute back to original shape
z_batched = z_batched.permute(0, 2, 1, 3).reshape(batch_size, seq_len, embed_dim)
# [batch_size, seq_len, embed_dim]

In [ ]:
Q_batched.shape

In [ ]:
torch.tril(torch.ones(seq_len, embed_dim))

In [ ]:
# With nn.Linear layers with learnable parameters
batch_size = 4
seq_len = 100
embed_dim = 512

num_heads = 8
head_dim = embed_dim // num_heads #64

X = torch.randn(batch_size, seq_len, embed_dim) #input

# weight matrices to get Qs, Ks, Vs projections
q_linear = nn.Linear(embed_dim,embed_dim)
k_linear = nn.Linear(embed_dim, embed_dim)
v_linear = nn.Linear(embed_dim, embed_dim)
out_linear = nn.Linear(embed_dim, embed_dim) # final project layer

# Q = X @ Wq (batched and boradcasted)
# [B, S, E] -> [B, S, H, D] -> [B, H, S, D]
Q_batched = q_linear(X).view(batch_size, seq_len, num_heads, head_dim).permute(0, 2, 1, 3)
K_batched = k_linear(X).view(batch_size, seq_len, num_heads, head_dim).permute(0, 2, 1, 3)
V_batched = v_linear(X).view(batch_size, seq_len, num_heads, head_dim).permute(0, 2, 1,3)

# Attention
# Q  [num_seq, head_dim]
# z = softmax(Q *K.t) V / sqrt(head_dim)
# batched matmul
S = torch.matmul(Q_batched, torch.transpose(K_batched,2,3))
S = S / math.sqrt(head_dim) #[batch_size, num_heads, seq_len, seq_len]

#mask
mask = torch.tril(torch.ones(seq_len, seq_len)).bool()
S = S.masked_fill(~mask, float('-inf'))

P = torch.softmax(S, dim=-1) #[batch_size, num_heads, seq_len, seq_len]
O = torch.matmul(P, V_batched) #[batch_size, num_heads, seq_len, head_dim]
# reshape/permute back to original shape
z_batched = O.permute(0, 2, 1, 3).reshape(batch_size, seq_len, embed_dim)
# [batch_size, seq_len, embed_dim]

# Final output layer
output = out_linear(z_batched)
print(output.shape)